In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/HireTrust/clean_jobs.csv")
df["text"] = df["text"].fillna("")
print(df.shape)
print(df["fraudulent"].value_counts())

(17880, 2)
fraudulent
0    17014
1      866
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["fraudulent"],
    test_size=0.2, stratify=df["fraudulent"], random_state=42)

print("Train:", len(X_train), "| Test:", len(X_test))
print("Fake % in train:", round(y_train.mean() * 100, 2))
print("Fake % in test: ", round(y_test.mean() * 100, 2))

Train: 14304 | Test: 3576
Fake % in train: 4.84
Fake % in test:  4.84


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, stop_words="english")
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print(X_train_vec.shape)

(14304, 20000)


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

lazy = DummyClassifier(strategy="most_frequent")
lazy.fit(X_train_vec, y_train)
print(classification_report(y_test, lazy.predict(X_test_vec), zero_division=0))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98      3403
           1       0.00      0.00      0.00       173

    accuracy                           0.95      3576
   macro avg       0.48      0.50      0.49      3576
weighted avg       0.91      0.95      0.93      3576



In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3403
           1       0.71      0.90      0.79       173

    accuracy                           0.98      3576
   macro avg       0.85      0.94      0.89      3576
weighted avg       0.98      0.98      0.98      3576

[[3340   63]
 [  18  155]]


In [ ]:
import numpy as np

words = np.array(tfidf.get_feature_names_out())
weights = model.coef_[0]
top = np.argsort(weights)[-20:][::-1]
for i in top:
    print(f"{words[i]:25s} {weights[i]:.2f}")

earn                      4.08
accion                    3.99
link                      3.86
entry                     3.81
phone                     3.31
email                     3.28
000                       3.21
aptitude                  3.17
money                     3.11
clerk                     3.06
aecom                     3.03
hospital                  2.89
administrative            2.86
subsea                    2.74
cash                      2.68
oil                       2.67
financing                 2.65
surgical                  2.65
offshore                  2.64
requirements              2.64


In [ ]:
sample = "work from home earn 50000 per week no experience pay registration fee contact on whatsapp"
vec = tfidf.transform([sample])
prob = model.predict_proba(vec)[0][1]
print("Fake probability:", round(prob * 100), "%")

Fake probability: 86 %


In [ ]:
import joblib

joblib.dump(model, "/content/drive/MyDrive/HireTrust/baseline_model.joblib")
joblib.dump(tfidf, "/content/drive/MyDrive/HireTrust/tfidf_vectorizer.joblib")
print("Saved")

Saved
